In [40]:
from transformers import AutoTokenizer, TFAutoModel
import tensorflow as tf

In [41]:
import numpy as np

In [42]:
import tensorflow.keras as keras

In [43]:
import pandas as pd

In [44]:
df=pd.read_csv('train.csv')
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [45]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['target'], test_size=0.2, random_state=42)

In [46]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [50]:
X_train.head()

4996    Courageous and honest analysis of need to use ...
3263    @ZachZaidman @670TheScore wld b a shame if tha...
4907    Tell @BarackObama to rescind medals of 'honor'...
2855    Worried about how the CA drought might affect ...
4716    @YoungHeroesID Lava Blast &amp; Power Red #Pan...
Name: text, dtype: object

In [55]:
X =df['text']
y=df['target']
X

0       Our Deeds are the Reason of this #earthquake M...
1                  Forest fire near La Ronge Sask. Canada
2       All residents asked to 'shelter in place' are ...
3       13,000 people receive #wildfires evacuation or...
4       Just got sent this photo from Ruby #Alaska as ...
                              ...                        
7608    Two giant cranes holding a bridge collapse int...
7609    @aria_ahrary @TheTawniest The out of control w...
7610    M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...
7611    Police investigating after an e-bike collided ...
7612    The Latest: More Homes Razed by Northern Calif...
Name: text, Length: 7613, dtype: object

In [56]:
embeddings = model.encode(X.tolist())

In [60]:
# Define a simple neural network
nn_model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(384,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # For binary classification
])

# Compile the model
nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
nn_model.fit(embeddings, y, epochs=50, batch_size=32)


Epoch 1/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 885us/step - accuracy: 0.7534 - loss: 0.5584
Epoch 2/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 882us/step - accuracy: 0.8181 - loss: 0.4171
Epoch 3/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 963us/step - accuracy: 0.8176 - loss: 0.4082
Epoch 4/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 869us/step - accuracy: 0.8285 - loss: 0.3880
Epoch 5/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8342 - loss: 0.3819
Epoch 6/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 890us/step - accuracy: 0.8313 - loss: 0.3768
Epoch 7/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 879us/step - accuracy: 0.8424 - loss: 0.3550
Epoch 8/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 883us/step - accuracy: 0.8548 - loss: 0.3347
Epoch 9/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8579 - loss: 0.3227  
Epoch 10/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 918us/step - accuracy: 0.8669 - loss: 0.3165
Epoch 11/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8635 - loss: 0.3115  
Epoch 12/50
238/238 ━━━━━━━━━━━━

In [82]:
df_new=pd.read_csv('test.csv')
df_new.shape

(3263, 4)

In [85]:
X_test_new=df_new['text']
X_test_new.shape

(3263,)

In [86]:
test_embeddings=model.encode(X_test_new.to_list())

In [87]:
test_embeddings.shape

(3263, 384)

In [88]:
predictions = nn_model.predict(test_embeddings)

102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 520us/step


In [89]:
predictions.shape

(3263, 1)

In [90]:
id=[i for i in df_new['id']]
pred=[]
for i in predictions:
    if i<0.5:
        pred.append(0)
    else:
        pred.append(1)

In [95]:
# Create a DataFrame to hold the results
output_df = pd.DataFrame({'id':id,'target':pred})

output_df

,id,target
0,0,0
1,2,1
2,3,0
3,9,1
4,11,1
...,...,...
3258,10861,1
3259,10865,1
3260,10868,1
3261,10874,1


In [96]:
# Save to CSV
output_df.to_csv('model_predictions.csv', index=False)

print("Predictions saved to 'model_predictions.csv'")

Predictions saved to 'model_predictions.csv'
